In [ ]:
from datasets import load_dataset

ds = load_dataset("Genius-Society/Pima")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

validation.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/614 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/77 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/77 [00:00<?, ? examples/s]

In [ ]:
!pip install pennylane datasets

import pennylane as qml
from pennylane import numpy as np
# Extract features and labels into arrays
X = np.array([[s['Pregnancies'], s['Glucose'], s['BloodPressure'], s['SkinThickness'],
               s['Insulin'], s['BMI'], s['DiabetesPedigreeFunction'], s['Age']]
              for s in ds['train']])
y = np.array([s['Outcome'] for s in ds['train']])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 934.3/934.3 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 34.8 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/pennylane/__init__.py:209: RuntimeWarning: PennyLane is not yet compatible with JAX versions > 0.6.2. You have version 0.7.2 installed. Please downgrade JAX to 0.6.2 to avoid runtime errors using python -m pip install jax~=0.6.0 jaxlib~=0.6.0
  warnings.warn(


In [ ]:
num_features = X.shape[1]       # 8 features
num_qubits = int(np.ceil(np.log2(num_features)))  # 3 qubits suffisent
dev = qml.device("default.qubit", wires=num_qubits)

X_norm = np.array([x / np.linalg.norm(x) for x in X])

def amplitude_circuit(x):
    qml.AmplitudeEmbedding(features=x, wires=range(num_qubits), normalize=False)
    return qml.state()
# Créer le QNode (circuit exécuté sur le simulateur)
qnode = qml.QNode(amplitude_circuit, dev)


# Exemple : encodage de la première ligne

first_sample = X_norm[0]
quantum_state = qnode(first_sample)

print("Première ligne normalisée :")
print(first_sample)

print("\nVecteur d'état quantique (encodage en amplitude) :")
for idx, amp in enumerate(quantum_state):
    state = format(idx, f'0{num_qubits}b')
    print(f"|{state}> : {amp}")


# Encoder tout le dataset

all_quantum_states = np.array([qnode(x) for x in X_norm])
print(f"\nForme du tableau d'états quantiques : {all_quantum_states.shape}")

Première ligne normalisée :
[0.01607941 0.68739482 0.2894294  0.11657573 0.62307718 0.17526558
 0.00192551 0.10451617]

Vecteur d'état quantique (encodage en amplitude) :
|000> : (0.016079411061816436+0j)
|001> : (0.6873948228926526+0j)
|010> : (0.2894293991126959+0j)
|011> : (0.11657573019816916+0j)
|100> : (0.6230771786453869+0j)
|101> : (0.17526558057379918+0j)
|110> : (0.0019255094746525182+0j)
|111> : (0.10451617190180684+0j)

Forme du tableau d'états quantiques : (614, 8)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1. Préparation des données
# On convertit en réel car les arbres de décision ne supportent pas les nombres complexes
X_final = np.real(all_quantum_states)

# 2. Découpage en données d'entraînement et de test
# On garde 20% des données pour vérifier si le modèle a bien appris
X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.2, random_state=42)

# 3. Création de l'arbre de décision
clf = DecisionTreeClassifier(max_depth=5, random_state=42)

# 4. Entraînement
clf.fit(X_train, y_train)

# 5. Calcul de la précision (Accuracy)
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"--- Résultat pour l'encodage Amplitude ---")
print(f"Accuracy : {acc * 100:.2f}%")

--- Résultat pour l'encodage Amplitude ---
Accuracy : 60.16%
